<p><small><small>This Notebook is made available subject to the licence and terms set out in the <a href = "http://www.github.com/google-deepmind/ai-foundations">AI Research Foundations Github README file</a>.</small></small></p>

<img src="https://storage.googleapis.com/dm-educational/assets/ai_foundations/GDM-Labs-banner-image-C7-white-bg.png">

# Lab: Prepare your Data for Building a Text Summarization Model

<a href='https://colab.research.google.com/github/google-deepmind/ai-foundations/blob/master/course_8/gdm_lab_8_5_summarization_part_1.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg' alt='Open In Colab'/></a>

2-4 hours (depending on the amount of data cleaning necessary)

In this first lab, you will load, clean and format the data for training a summarization model. At this stage, you should have completed previous activities from the Capstone course. Specifically you should have:

1. Defined your summarization task,
2. Downloaded or constructed your dataset with the original text and the summaries,
3. Documented your dataset with a Data Card, and
4. Assessed the impact of your summarization model.

In this and the next lab, you will build on the activities to turn your project plan into a working summarization model.

Similarly, as in the previous activities, this lab also contains a worked example for every step. Consider the code for the worked example as a template for your own implementation. Note, however, that some steps may not be applicable to your project, and likewise, you may have to complete additional steps, so treat the template in this and the next lab just as a starting point and feel free to add or delete code cells to this notebook as you see fit.


## Overview

In this first part of implementing your capstone project, you will focus on preparing your data for training a text summarization model.

### What you will learn

By the end of this first part of the capstone project, you will be able to:

* Clean and preprocess text data to prepare quality inputs for fine-tuning instruction-based conversational models.

* Split datasets into training and test sets.

* Format the conversational data so that it can be used to fine-tune an LLM.

### Tasks

In this lab, you will:

* Load the dataset that you prepared in previous actitivites.

* Clean individual datapoints, if necessary.

* Split the dataset into training and test sets.

* Format the data and prepare it for LLM fine-tuning.

* Save the dataset in a format that is compatible with the machine learning pipeline.

The end result of this lab will be a processed dataset that you can use in the second part to train and evaluate your text summarization model.

### What you will use

This lab builds on materials from several previous courses. If you would like to refresh your knowledge on any of the following topics, you can find the relevant materials in the following courses:

- **General knowledge on LLMs and transformers** (Courses 01 Build Your Own Small Language Model, 03 Design and Train Neural Networks, and 04 Disccover The Transformer Architecture).

- **Data preparation** techniques (Course 02 Represent Your Text Data).

- **Formatting data for fine-tuning** (Course 05 Fine-Tune Your Model).

- **Splitting data** techniques (Course 03 Training a Neural Network).


## How to use Google Colaboratory (Colab)

Google Colaboratory (also known as Google Colab) is a platform that allows you to run Python code in your browser. The code is written in cells that are executed on a remote server.

To run a cell, hover over a cell, and click on the `run` button to its left. The run button is the circle with the triangle (▶). Alternatively, you can also click on a cell and use the keyboard combination Ctrl+Return (or ⌘+Return if you are using a Mac).

To try this out, run the following cell. This should print today's day of the week below it.

In [ ]:
from datetime import datetime
print(f"Today is {datetime.today():%A}.")

Note that the order in which you run the cells matters. When you are working through a lab, make sure to always run all cells in order. Otherwise, the code might not work. If you take a break while working on a lab, Colab may disconnect you and in that case, you have to execute all cells again before  continuing your work. To make this easier, you can select the cell you are currently working on and then choose __Runtime → Run before__  from the menu above (or use the keyboard combination Ctrl/⌘ + F8). This will re-execute all cells before the current one.

### Using Colab with a GPU

Follow these steps to run the activities in this lab on a GPU:

1.  In the top menu bar, click on **Runtime**.
2.  Select **Change runtime type** from the dropdown menu.
3.  In the pop-up window under **Hardware Accelerator**, select **GPU** (usually listed as `T4 GPU`).
5.  Click **Save**.

Your Colab session will now restart with GPU access.

Note that access to GPUs is limited and at times, you may not be able to run this lab on a GPU. All activities will still work but they will run slower and you will have to wait longer for some of the cells to finish running.


## Imports


The following cell contains imports for a range of libraries that you may use for preparing your dataset. Depending on your implementation, you may need to import additional libraries.

In [ ]:
# Standard library imports.
import io # Input/output operations.
import json # JSON data handling.
import os # Operating system interface.
import re # Regular expressions for text processing.
import unicodedata # Unicode character database.
from textwrap import fill # Text wrapping utilities.
from typing import Any, Dict, Tuple, List

# Third-party data and utility libraries.
import numpy as np # Numerical computing.
import pandas as pd # Data manipulation and analysis.
from tqdm import tqdm # Progress bars.

from sklearn.model_selection import train_test_split # Data splitting utils.
import matplotlib.pyplot as plt # Plotting and visualisation.

# Deep learning and AI libraries.
import jax.numpy as jnp # JAX numerical computing.
import keras # High-level neural networks API.
import keras_nlp # Keras NLP extensions.

# Google Colab specific.
from google.colab import drive  # Access to Google Drive.
from google.colab import userdata  # Access to Colab secrets.

## Load your data

As a first step, you will have to load the dataset that you created in the previous activity, so that you can preprocess your data. One method to do this is to upload the dataset to Google Drive and then connect your Google Drive to Colab.

### Connect Colab with Google Drive

The following cell mounts your Google Drive so that you can access files by adding `/content/drive/MyDrive` to the beginning of your file path. Running this cell will open a dialog that asks you whether you want to connect Colab to Google Drive. Follow the instructions and enable all permissions, so that Colab can read and write files from Google Drive.

In [ ]:
drive.mount('/content/drive')

## Activity 1: Load and inspect your data


------
> 💻 **Your task:**
>
> Load and inspect your dataset by adding your code to the subsequent cells:
>
> 1. **Define the path to your dataset**: Set the path to your own dataset file path.
>
> 2. **Choose a data loading method** to match your data format:
>    - For CSV files: `pd.read_csv(data, encoding="utf-8")`.
>    - For JSON files: Use `json.load()` or `pd.read_json(data)`.
>    - For JSONL files: Use `pd.read_json(data, lines=True)`.
>    - For API data: Implement your own loading function.
>
> 3. **Verify your data structure**: After loading, check that:
>    - Your original texts are in an appropriate column.
>    - Your summaries are appropriately formatted.
>    - The encoding is preserved (especially for non-English text).
>
> 4. **Inspect the data**: Use `df.head()`, `df.info()`, and `df.columns` to understand your dataset structure and size.
>    - Identify the prompt and response columns.
>
>
> The goal of these steps is to bring all records into a variable (a Pandas `DataFrame`) so that you can explore the structure, check the columns, preview a few rows, and confirm the encoding, before moving on to transformation for LoRA training.
------


#### Worked example: Load and inspect your data


```python
# Open the file from Google Drive.
filepath = "/content/drive/MyDrive/africa_galore_summarization.jsonl"
# Read with pandas.
df = pd.read_json(filepath, lines=True)
# Other file-type examples:
# df = pd.read_json(filepath)
# df = pd.read_csv(filepath, encoding="utf-8")
# df = json.load(open('your_data.json'))
# df = load_from_api()

# Take a quick look at the first few rows.
display(df.head())

print("Columns:", df.columns.tolist())
print("Number of rows:", len(df))
```

#### Your implementation


In [ ]:
# Add your code here.


### Understanding the dataset

Before proceeding with data preprocessing and model training, it is a good idea to thoroughly examine the dataset. With large datasets, hidden issues such as duplicates, extreme text lengths, or imbalanced categories can significantly impact model performance. The following checks help you understand the structure and quality of your data, enabling informed decisions about cleaning and filtering.

The first step is to compute basic word count statistics for the text fields. Check out the worked example on how to calculate the minimum, maximum, and average word counts for any DataFrame column, which provides a quick overview of text length distributions.

#### Worked example: Compute word count statistics

```python
def compute_word_count_stats(df_column):
    """
    Compute minimum, maximum, and average word counts for a DataFrame column.

    Args:
      df_column (pd.Series): A pandas Series containing text data.

    Returns:
      tuple: A tuple of (min_words, max_words, avg_words), where:
        - min_words (int): Minimum number of words in the column
        - max_words (int): Maximum number of words in the column
        - avg_words (float): Average number of words in the column
    """
    # Drop missing values and ensure all entries are strings
    word_counts = (
        df_column
        .dropna()
        .astype(str)
        .str.split()
        .str.len()
    )

    return word_counts.min(), word_counts.max(), word_counts.mean()
```

#### Your implementation


In [ ]:
# Add your for analyzing your data here.


You can now apply the word count statistics function to each of the key text fields in the dataset. Understanding the word count distribution is useful for determining appropriate model parameters such as `sequence_length`, and for identifying any outliers or anomalies in the data. Significant differences in word counts between the source text (`expanded`) and target summary (`summary`) confirm the summarization task is meaningful.

#### Worked example: Print statistics


```python
min_w, max_w, avg_w = compute_word_count_stats(df["expanded"])
print(f"Expanded -> Min: {min_w}, Max: {max_w}, Avg: {avg_w:.2f}")

min_w, max_w, avg_w = compute_word_count_stats(df["description"])
print(f"Description -> Min: {min_w}, Max: {max_w}, Avg: {avg_w:.2f}")

min_w, max_w, avg_w = compute_word_count_stats(df["summary"])
print(f"Summary -> Min: {min_w}, Max: {max_w}, Avg: {avg_w:.2f}")
```

#### Your implementation


In [ ]:
# Add your code for printing dataset statistics here.


To gain further insight into the text length characteristics, it is helpful to examine the full distribution of word counts rather than just the summary statistics. Check out the worked example for how to compute additional statistics that give you an overview over the distribution of example lengths.

#### Worked example: More statistics

The following helper functions compute word counts for a given column and group them into bins, making it easier to visualise how entries are distributed across different length ranges. This can reveal whether the data is evenly spread or contains concentrations at particular lengths that may influence model training.


```python
def word_counts(series: pd.Series) -> pd.Series:
    """
    Compute word counts for each entry in a pandas Series.

    Args:
      series: A pandas Series containing text data.

    Returns:
      A pandas Series with the word count for each entry.
    """
    return (
        series
        .dropna()
        .astype(str)
        .str.split()
        .str.len()
    )


def word_count_distribution(
    series: pd.Series,
    bin_size: int = 10
) -> pd.DataFrame:
    """Compute the distribution of word counts grouped into bins.

    Groups word counts into ranges (e.g., 0-10, 10-20) and counts
    how many entries fall into each range. Useful for visualizing
    the spread of text lengths in a dataset.

    Args:
      series: A pandas Series containing text data.
      bin_size: Size of each word count bin.

    Returns:
      A DataFrame with columns 'word_range' and 'count'.
    """
    # Compute word counts for the series.
    wc = word_counts(series)

    # Create bins from 0 to max word count.
    max_words = wc.max()
    bins = np.arange(0, max_words + bin_size, bin_size)

    # Create labels for each bin range.
    labels = [
        f"{bins[i]}-{bins[i+1]}"
        for i in range(len(bins) - 1)
    ]

    # Bin the word counts and count entries in each bin.
    dist = (
        pd.cut(wc, bins=bins, labels=labels, right=False)
        .value_counts()
        .sort_index()
        .reset_index()
    )

    dist.columns = ["word_range", "count"]
    return dist

# Print distributions for each field in the dataset.
for col in ["expanded", "description", "summary"]:
    print(f"\nWord count distribution for '{col}':")
    print(word_count_distribution(df[col]))
```




#### Your implementation


In [ ]:
# Add your code to compute distributional statistics here.


### Finding repetitive data

With a large dataset, it is good to know how the data is distributed across different categories and item names, and especially if there are any duplicate entries. Duplicate content in the training data can lead to overfitting, where the model overfits to specific training instances rather than learning generalizable patterns. This can be problematic for summarization tasks, where the goal is for the model to learn how to condense diverse content rather than reproduce previously seen outputs.

Check out the worked example for how to compute whether any examples appear multiple times.

#### Worked example: Find duplicates



The following code checks for duplicate values in the three key text fields: `expanded` (the full source text), `description` (the shorter description), and `summary` (the target summary). This helps identify whether any content appears multiple times in the dataset.

```python
duplicate_counts = {
    "expanded": df.duplicated(subset=["expanded"]).sum(),
    "description": df.duplicated(subset=["description"]).sum(),
    "summary": df.duplicated(subset=["summary"]).sum(),
}

print("Number of duplicate rows per field:")
for field, count in duplicate_counts.items():
    print(f"{field}: {count}")
```

#### Your implementation


In [ ]:
# Add your code to find duplicates in your dataset here.


### Filter and balance the dataset

If you are using an existing dataset, you may want to filter the examples so that you have dataset with a more manageable size and balanced categories. For example, in the worked example, the dataset contains many thousands of examples across many different categories. For the purpose of this capstone, it can be better to filter the dataset so as to shorten fine-tuning times.


---------
> 💡 **Why balance the dataset?**
>
> If you choose to train your summarization model on multiple categories, having roughly equal numbers of examples per category helps prevent the model from learning summarization patterns biased towards the majority category. For example, if one category has significantly more examples than others, the model may learn to favour the writing style or content structure of that category. Capping each category to the same number of items ensures the model receives balanced exposure to all content types during fine-tuning, leading to more consistent summarization quality across different domains.
---------

#### Worked example: Filter the dataset


The Africa Galore dataset contains many thousand examples. With a dataset of this size, targeted filtering helps produce training data that is well suited to the summarization task. The code below restricts the data to selected categories and applies word-count ranges to both the source text and the target summary. This introduces consistency across training examples and keeps the task well defined, since source texts should be substantially longer than their summaries.

The code also removes rows with duplicate summaries, reducing the risk of the model overfitting to identical outputs during training. To maintain balance, the number of examples per story name is capped, preventing any single topic from dominating the dataset. A total row limit is then applied to keep training time and computational costs manageable. Finally, the data is shuffled to randomize example order and reduce the likelihood of the model learning artefacts from the original ordering.

The filtering parameters can be adjusted to match different dataset characteristics or training objectives. The ranges used in this example are informed by the earlier analysis of the dataset and are considered well suited given both the distribution of the data and the available processing constraints, such as GPU memory and training efficiency. In this context, summaries of 20-30 words and expanded story texts of 250-320 words represent a practical and balanced choice for the purposes of the worked example.

```python
# Define the categories to keep.
selected_categories = [
    "Story"
]

# Define filtering parameters.
max_per_name = 50  # Maximum examples per story name.
max_total_rows = 700  # Maximum total rows after all filtering.
expanded_word_range = (250, 320)  # Min and max words for 'expanded'.
summary_word_range = (20, 30)  # Min and max words for 'summary'.

# Create a working copy of the dataframe.
df = df.copy()

# Compute word counts for the 'expanded' and 'summary' fields.
df["expanded_word_count"] = (
    df["expanded"]
    .dropna()
    .astype(str)
    .str.split()
    .str.len()
)

df["summary_word_count"] = (
    df["summary"]
    .dropna()
    .astype(str)
    .str.split()
    .str.len()
)

# Filter the dataframe by selected categories only.
# Keep 'expanded' and 'summary' word count within specified range.
df = df[
    df["category"].isin(selected_categories) &
    (df["expanded_word_count"] >= expanded_word_range[0]) &
    (df["expanded_word_count"] <= expanded_word_range[1]) &
    (df["summary_word_count"] >= summary_word_range[0]) &
    (df["summary_word_count"] <= summary_word_range[1])
].copy()

# Drop any duplicate summaries to avoid training on identical targets.
df = df.drop_duplicates(subset=["summary"], keep="first")

# Cap to max_per_name rows per story name.
df = (
    df.sort_index()  # Preserve original ordering before sampling.
      .groupby("name", group_keys=False)
      .head(max_per_name)
)

# Cap to max_total_rows overall.
if len(df) > max_total_rows:
    df = df.head(max_total_rows)

# Shuffle the data to reinforce random ordering during training later.
# This prevents the model from learning spurious patterns based on data order.
# random_state ensures reproducibility.
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Validate the filtered data.
print("Filtered dataset:")
display(df.head())

print("\nColumns:", df.columns.tolist())
print("Number of rows:", len(df))
print("Unique categories:", df["category"].unique())
print("Unique story names:", df["name"].nunique())
```

#### Your implementation


In [ ]:
# Add your code for filtering your dataset here (if applicable).


## Activity 2: Clean the data



------
> 💻 **Your task:**
>
> Clean and prepare your text data for fine-tuning:
>
> 1. **Assess your data quality**: Examine your text data for:
>    - HTML tags, markup, or special formatting.
>    - Non-text characters (emojis, symbols, special Unicode).
>    - Inconsistent encoding or corrupted characters.
>    - Excessive whitespace or formatting issues.
>
> 2. **Apply appropriate cleaning functions**:
>    - Use `clean_html()` to remove HTML tags and entities.
>    - Use `clean_unicode()` to remove emojis and non-text symbols.
>    - Add custom cleaning functions if needed for your domain.
>>
> 3. **Preview cleaning results**: Compare before/after examples to ensure:
>    - Important content is preserved.
>    - Unwanted elements are removed.
>    - Text remains readable and coherent.
>
> 4. **Decide on final text column**: Choose whether to:
>    - Replace original text with cleaned version.
>    - Keep both versions for comparison.
>    - Apply different levels of cleaning for different purposes.
>
> **Quality check**: Examine several examples to verify that cleaning improves data quality without losing essential information.
>
------

Refer to Course 02 Represent Your Text Data for examples on how to adjust and clean your data.

The following code provides a template using functions from Course 02 Represent Your Text Data to clean text fields in your dataframe.

In [ ]:
def clean_html(text: str) -> str:
    """Strip basic HTML markup and common entities from a string.

    The funcion does **not** attempt full HTML parsing; for more complex markup
    consider ``BeautifulSoup`` or ``html.unescape``.

    Args:
      text: The text string that may contain HTML tags or entities.

    Returns:
      A cleaned string with tags stripped and the entities '&nbsp;', '&amp;',
        '&lt;' and '&gt;' converted to ' ', '&', '<' and '>'.
    """

    # Remove HTML tags.
    text = re.sub(r"<.*?>", "", text)

    # Replace HTML entities.
    text = re.sub("&nbsp;", " ", text)  # Replace non-breaking space.
    text = re.sub("&amp;", "&", text)  # Replace "&amp;" with "&".
    text = re.sub("&lt;", "<", text)  # Replace "&lt;" with "<".
    text = re.sub("&gt;", ">", text)  # Replace "&gt;" with ">".

    return text


def clean_unicode(text: str) -> str:
    """Removes non-text unicode characters from a string.

    Args:
      text: The original text which may contain special characters.

    Returns:
      The input text string with emojis and other non-text symbols removed.
    """

    categories_to_keep = {"L", "N", "P"}  # L=letters, N=numbers, P=punctuation.

    keep = []
    for ch in text:
        do_keep = ch.isspace()  # Preserve spaces.
        if not do_keep:
            for category in categories_to_keep:
                if unicodedata.category(ch).startswith(category):
                    do_keep = True
                    break
        if do_keep:
            keep.append(ch)
    return "".join(keep)

You can now apply the data cleaning functions to your data.

#### Worked example: Clean the target text column and preview before/after.


```python
# Clean the target text column and preview before/after.
# In this example, 'description' is the primary input for summarization.
# If your descriptions contain HTML or special characters, apply cleaning here.
# For extended pipelines using 'expanded', adjust the column name.

# Keep an original copy for reference.
df["expanded_original"] = df["expanded"].astype(str)
df["expanded_clean"] = (
    df["expanded_original"].map(clean_html).map(clean_unicode)
)

# Quick preview.
preview_cols = [
    c
    for c in [
        "category",
        "expanded_original",
        "expanded_clean",
    ]
    if c in df.columns
]
display(df[preview_cols].head(5))

# Optionally, replace the original name with the cleaned version here.
df["expanded"] = df["expanded_clean"]
```

#### Your implementation


In [ ]:
# Add your code here for applying data cleaning here.

## Activity 3: Split the original data



------
> 💻 **Your task:**
>
> Create a separate test set for evaluation:
>
> Create training (`df_train`) and test (`df_test`) sets from your filtered data. The code in the worked example uses stratified splitting to maintain category proportions in both sets, ensuring fair evaluation of the summarization model. You may be able to adapt this code for your dataset.
>
> - **Important considerations**:
>   - **Performance warning**: Gemma3-1B may perform poorly on completely unknown test examples if your training dataset is small.
>   - For small datasets, consider using more training data and smaller test sets.
>   - Ensure evaluation data reflects your actual use case and domain.
>
------

Building a summarization model introduces evaluation challenges and makes evaluation inherently subjective, requiring qualitative methods such as human assessment and manual spot-checking for content coverage, faithfulness to the source, conciseness, and fluency.

You may recall from Course 03 Design and Train Neural Networks that it is standard to split datasets into three splits: one for training, one for validation, and one for testing. For this capstone in which you build a model that produces very open-ended answers, however, you will likely not be able to use any automatic metrics and therefore, having a validation set will likely be of limited value. For this reason, the worked example splits the data only into a training and test split and you may want to do the same for your dataset.

### Create a test set from the filtered data

With the dataset already filtered and balanced, it can be split directly into training and test sets. In the next code cell, implement the splitting of the dataset.



#### Worked example: Split the dataset for testing


```python
# Split 20% of the filtered dataframe for testing.
test_fraction = 0.2

# Use stratified splitting only when there is more than one category.
# If there is a single category, stratification is unnecessary.
stratify_col = df["category"] if df["category"].nunique() > 1 else None

df_train, df_test = train_test_split(
    df,
    test_size=test_fraction,
    random_state=42,
    stratify=stratify_col
)

# Reset indices for clean dataframes.
df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# Update df to be the training set for downstream use.
df = df_train.copy()

# Validate the split.
print(f"Training set: {len(df_train)} items")
print(f"Test set: {len(df_test)} items ({test_fraction:.0%} of original)")
print("\nTest set items per category:")
print(df_test["category"].value_counts())
display(df_test.head())
```

#### Your implementation


In [ ]:
# Add your code for splitting the dataset here.


-----
> **ℹ️ Info: Stratified splitting**
>
> Stratified splitting ensures that the proportion of samples from each category is preserved in both the training and test sets. While stratification is often associated with classification tasks, it is equally valuable for summarization when your dataset contains multiple content types or topics.
>
> For example, if your summarization dataset contains stories from different cultural regions or thematic categories, stratified splitting ensures both training and test sets include representative samples from each. This prevents situations where the model trains primarily on one type of content but is evaluated on another.
>
> For summarization specifically, balanced representation across categories helps the model learn diverse summarization patterns and ensures the evaluation fairly reflects performance across all content types in your dataset.
>
> For more information, see  [scikit-learn documentation on stratification](https://scikit-learn.org/stable/modules/cross_validation.html#stratification).
>
-----

## Activity 4: Format the data



------
> 💻 **Your task:**
>
>Convert your dataset into a format that can be used to fine-tune a Gemma model.
>
> 1. **Define a `to_instruction_record` function**. This function should turn an example from your dataset into a pair of input and output that can be used to fine-tune a Gemma model.
>
> 2. **Design your instruction prompt**: Create a clear, specific instruction that tells the model what to do:
>    - Example: "Summarize the following text: ".
>    - Example: "Create a one-sentence summary of the following text:".
>
> 3. **Test the transformation**: Run the code to transform your data and inspect the first record to ensure it looks correct.
>
> 4. **Save your training data**: Save your data to Google Drive for fine-tuning.
>
> **Verification**: Check that your JSONL file has the correct format with `input` and `output` fields for each record.
>
> Take a look at the worked example, which implements all of these steps and adapt it to your dataset
------


#### Worked example: Prepare examples for instruction-tuning

```python
def to_instruction_record(row) -> dict[str, str]:
    """
    Transform a single row of the original dataset
    into instruction-tuning format for summarization.

    Args:
      row: A single row from a Pandas DataFrame.

    Returns:
      dict: A dictionary with "input" and "output" as keys.
    """

    return {
        "input": "Summarize: " + row["expanded"],
        "output": row["summary"],
    }


# Apply the transformation.
instruction_records_train = [
    to_instruction_record(r) for _, r in df_train.iterrows()
]
instruction_records_test = [
    to_instruction_record(r) for _, r in df_test.iterrows()
]

# Preview a single transformed record.
print(json.dumps(instruction_records_train[0], ensure_ascii=False, indent=2))

# Save to a JSONL file for fine-tuning.
with open(
    "/content/drive/MyDrive/summarization_training_data.jsonl",
    "w",
    encoding="utf-8"
) as f:
    for rec in instruction_records_train:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

with open(
    "/content/drive/MyDrive/summarization_test_data.jsonl",
    "w",
    encoding="utf-8"
) as f:
    for rec in instruction_records_test:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

```

#### Your implementation


In [ ]:
# Add your code for converting and saving your data here.


## Summary

In the first part of this capstone pathway, you prepared the dataset so that you can use it for fine-tuning an LLM to act as a summarization model. This involved loading the dataset, cleaning the data, splitting it into a training, development, and test split, and formatting it to be used for fine-tuning.

In the second part of this captstone pathway, you will use these preprocessed data splits to fine-tune a model for summarization.
